<a href="https://colab.research.google.com/github/AzizullahMemonAi/FlyRank-ML-Assignments/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AzizullahMemonAi/FlyRank-ML-Assignments/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

In [10]:
from google.colab import drive
import pandas as pd

drive.mount('/content/drive')

file_path = "/content/drive/MyDrive/FlyRank Dataset/content_refresh_anonymized.csv"

df = pd.read_csv(file_path)

print("Dataset loaded successfully")
print("Full dataset shape:", df.shape)

print("\nAvailable columns:")
print(df.columns.tolist())

if "month" in df.columns:

    # March 2026 slice
    march_df = df[df["month"].astype(str) == "2026-03"].copy()

    row_count = len(march_df)
    unique_content = march_df["content_id"].nunique()
    duplicate_count = row_count - unique_content

    print("\n--- March 2026 Verification ---")
    print("March rows:", row_count)
    print("Unique content IDs:", unique_content)
    print("Duplicate content rows:", duplicate_count)

    if duplicate_count == 0:
        print("Verified: one row = one content item for March 2026")
    else:
        print("Warning: duplicate content IDs exist in this slice")

else:
    print("\n'month' column is NOT present in this dataset.")
    print("This means this starter CSV cannot directly verify the March 2026 warehouse slice.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Dataset loaded successfully
Full dataset shape: (30000, 44)

Available columns:
['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct']

'month' column is N

## 1. Unit of analysis + time window

For my Lane 3 clustering task, **one row represents one content item/page for one monthly snapshot**.

I will use **March 2026 (`2026-03`)** as my development time window. March is a mid-panel month, which allows me to explore the data without using the final June 2026 month that should remain separate for later evaluation.

* **Unit of analysis:** One content item/page
* **Time window:** March 2026
* **Expected grain:** One row per content item in the March 2026 monthly snapshot

I will verify this grain below by comparing the total number of rows with the number of unique content items in the March slice.


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

In [11]:
field_buckets = {
    "features": [
        "impressions_90d",
        "ctr",
        "avg_position",
        "engagement_rate",
        "content_age_days"
    ],

    "label": [
        # No training label for unsupervised clustering
    ],

    "context": [
        "content_id",
        "month",
        "content_type"
    ],

    "excluded": [
        "client_id",
        "future_or_label_derived_fields",
        "june_2026_outcomes"
    ]
}

for bucket, fields in field_buckets.items():
    print(f"\n{bucket.upper()}:")
    if fields:
        for field in fields:
            print(" -", field)
    else:
        print(" - None (unsupervised clustering)")


FEATURES:
 - impressions_90d
 - ctr
 - avg_position
 - engagement_rate
 - content_age_days

LABEL:
 - None (unsupervised clustering)

CONTEXT:
 - content_id
 - month
 - content_type

EXCLUDED:
 - client_id
 - future_or_label_derived_fields
 - june_2026_outcomes


## 2. Fields: feature / label / context / excluded

For my Lane 3 clustering task, I will organize the fields into four groups:

### Features

These are the numerical fields I plan to use as inputs for clustering:

* `impressions_90d` — measures search visibility.
* `ctr` — measures the observed click-through rate.
* `avg_position` — describes observed search position.
* `engagement_rate` — describes user engagement.
* `content_age_days` — describes how old the content is.

I will keep the clustering feature set small and interpretable, with a maximum of five features.

### Label

My main clustering task has **no predefined training label** because it is an unsupervised learning problem.

The model's output will be a cluster assignment for each content item rather than a predicted class.

Fields such as `trend_direction` may be inspected later as descriptive information, but they will not be used as the target for training the clustering model.

### Context

These fields help identify or describe the observation but will not be used directly as clustering features:

* `content_id` — identifies the content item and helps verify the grain.
* `month` — identifies the monthly snapshot and allows me to select March 2026.
* `content_type` — may help describe the discovered clusters after clustering.

### Excluded

I will deliberately exclude:

* `client_id` — it identifies the client and is not needed for discovering page-performance archetypes.
* Future or label-derived fields — these could introduce data leakage because they may contain information that would not be available at the decision moment.
* Raw identifiers such as `content_id` — although needed as context, they should not be numerical inputs to the clustering algorithm.
* The final June 2026 outcome information — June is being kept separate rather than used during development.

This separation helps ensure that the clustering model uses only appropriate, measurable features and does not accidentally learn from identifiers, private information, or future information.


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [12]:
# Section 3: Verify data contract

print("=== GRAIN AND COUNTS ===")
print("Total rows:", len(df))
print("Unique content items:", df["content_id"].nunique())
print("\n=== MISSING VALUES ===")

features = [
    "impressions_90d",
    "ctr",
    "avg_position",
    "engagement_rate",
    "content_age_days"
]

print(df[features].isnull().sum())

print("\n=== OBSERVED WINDOWS ===")
print("Impressions window: 90 days")
print("Clicks window: 90 days")
print("Sessions window: 90 days")


print("\n=== SAMPLE ===")
display(df[["content_id"] + features].head())

=== GRAIN AND COUNTS ===
Total rows: 30000
Unique content items: 30000

=== MISSING VALUES ===
impressions_90d     0
ctr                 0
avg_position        0
engagement_rate     0
content_age_days    0
dtype: int64

=== OBSERVED WINDOWS ===
Impressions window: 90 days
Clicks window: 90 days
Sessions window: 90 days

=== SAMPLE ===


,content_id,impressions_90d,ctr,avg_position,engagement_rate,content_age_days
0,content_304f48230142,3803,0.76,10.6,5.88,187
1,content_a1fb4e703a9e,15320,0.05,20.3,0.00,445
2,content_9aa793d4d895,12581,0.09,36.5,0.00,141
3,content_331d6c4de07b,11751,0.49,6.2,1.28,463
4,content_d99b7a2d90ca,19140,0.13,44.0,0.00,263


### Verification

I checked the dataset grain, row counts, missing values, and measurement windows before using the fields for my clustering task.

The grain check compares the total number of rows with the number of unique `content_id` values to understand whether each row represents a unique content item.

I also checked missing values for my five planned clustering features: `impressions_90d`, `ctr`, `avg_position`, `engagement_rate`, and `content_age_days`.

The search-performance features use observed historical windows, such as the previous 90 days. These checks help me understand what information is actually available before building the clustering feature frame.


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

In [13]:
# Section 4: Data Limits

print("=== DATA LIMIT CHECK ===")

# 1. Dataset size
print("\nTotal rows:", len(df))
print("Total columns:", len(df.columns))

# 2. Check content age variation
print("\n=== CONTENT HISTORY ===")
print("Minimum content age:", df["content_age_days"].min(), "days")
print("Maximum content age:", df["content_age_days"].max(), "days")
print("Median content age:", df["content_age_days"].median(), "days")

# 3. Check missing values in important Lane 3 features
features = [
    "impressions_90d",
    "ctr",
    "avg_position",
    "engagement_rate",
    "content_age_days"
]

print("\n=== MISSING VALUES ===")
print(df[features].isnull().sum())

# 4. Show measurement-window columns
window_columns = [
    col for col in df.columns
    if "30d" in col or "90d" in col
]

print("\n=== WINDOW-BASED FEATURES ===")
for col in window_columns:
    print("-", col)

=== DATA LIMIT CHECK ===

Total rows: 30000
Total columns: 44

=== CONTENT HISTORY ===
Minimum content age: 90 days
Maximum content age: 564 days
Median content age: 236.0 days

=== MISSING VALUES ===
impressions_90d     0
ctr                 0
avg_position        0
engagement_rate     0
content_age_days    0
dtype: int64

=== WINDOW-BASED FEATURES ===
- impressions_90d
- clicks_90d
- pageviews_90d
- sessions_90d
- users_90d
- engaged_sessions_90d
- ai_sessions_90d
- scroll_events_90d
- impressions_last_30d
- clicks_last_30d
- sessions_last_30d
- impressions_prev_30d
- clicks_prev_30d
- sessions_prev_30d


### Data-limit observation

The checks show that content items can have different amounts of history and that several performance features are calculated using rolling 30-day or 90-day windows. These windows may overlap across observations, so nearby monthly observations should not automatically be treated as fully independent.

I also checked missing values in the five planned clustering features because availability may differ across signals and time periods.

These checks describe limitations of the observed data. They do not establish causal relationships. Therefore, I will use the data to discover **observed and directional patterns for decision-support**, rather than claiming that a cluster or individual feature causes content performance.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.